# Notebook 01 — Data Exploration

Before building any model, I spent a lot of time just staring at the MADAR corpus. This notebook documents that process — the class distribution issues, the text length variability, and the dialect examples that shaped my intuitions about what the model would need to handle.

If you don't have MADAR access yet, this notebook falls back to a **synthetic dataset** that has the same structure, so all the code runs end-to-end without any data download.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})

DIALECT_NAMES = ['Gulf', 'Egyptian', 'Levantine', 'Maghrebi', 'Iraqi']
COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
print('Imports OK')

## 1. Load the dataset

We try to load the real MADAR-processed CSV first. If it's not there, we generate synthetic data that mimics the corpus structure (5 dialect classes, similar text length distribution). The synthetic data is clearly labeled so you know which mode you're in.

In [ ]:
from pathlib import Path

DATA_IS_SYNTHETIC = False
csv_path = Path('../data/processed/train.csv')

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f'Loaded real MADAR data: {len(df):,} training examples')
else:
    DATA_IS_SYNTHETIC = True
    print('MADAR not found — generating synthetic corpus for demo...')

    # Synthetic sentences with genuine dialectal vocabulary
    dialect_sentences = {
        0: [  # Gulf
            'شلونك اليوم كيف صحتك', 'وين رايح بكره', 'ما قدرت اجي امبارح',
            'هذا الشي مو صح', 'يالله نمشي الحين', 'زين جدا والله',
            'وش تبي', 'ما ابي ارجع بدري', 'شوف هذا الكتاب', 'اهلين وسهلين'
        ],
        1: [  # Egyptian
            'ازيك النهارده', 'عامل ايه يا صاحبي', 'انت فين دلوقتي',
            'ده مش كويس خالص', 'يلا بينا بسرعة', 'تمام اوي والله',
            'عايز ايه', 'مش هروح بدري', 'بص على الكتاب ده', 'اهلا وسهلا'
        ],
        2: [  # Levantine
            'كيفك هلق شو بدك', 'وين رايح بكرا', 'ما قدرت اجي مبارح',
            'هيدا مش منيح', 'يلا نروح هلق', 'منيح كتير والله',
            'شو بدك', 'ما بدي روح بدري', 'شوف هالكتاب', 'اهلا وسهلا'
        ],
        3: [  # Maghrebi
            'كيداير واش خبارك', 'فين غادي غدوة', 'ما قدرتش نجي لبارح',
            'هادا مشي مزيان', 'يلا نمشيو دروك', 'مزيان بزاف والله',
            'اش خصك', 'ما بغيتش نمشي بكري', 'شوف هاد الكتاب', 'مرحبا بيك'
        ],
        4: [  # Iraqi
            'شلونك اليوم شكو ماكو', 'وين رايح باجر', 'ما كدرت اجي البارحة',
            'هذا مو زين', 'يالله نروح هسة', 'زين كلش والله',
            'شكو تريد', 'ما ريد اروح بدري', 'شوف هذا الكتاب', 'اهلا وسهلا'
        ]
    }

    rng = np.random.default_rng(42)
    rows = []
    # Introduce realistic class imbalance: Gulf < others
    class_sizes = {0: 800, 1: 1200, 2: 1100, 3: 950, 4: 1050}
    for label, size in class_sizes.items():
        sents = dialect_sentences[label]
        for _ in range(size):
            # combine 1-3 sentences
            n_sents = rng.integers(1, 4)
            idxs = rng.integers(0, len(sents), size=n_sents)
            text = ' '.join(sents[i] for i in idxs)
            rows.append({'text': text, 'dialect_id': label,
                         'dialect': DIALECT_NAMES[label]})
    df = pd.DataFrame(rows)
    print(f'Synthetic corpus: {len(df):,} examples')

if 'dialect' not in df.columns:
    df['dialect'] = df['dialect_id'].map(dict(enumerate(DIALECT_NAMES)))

df['n_words'] = df['text'].str.split().str.len()
df['n_chars'] = df['text'].str.len()
df.head()

## 2. Class distribution

Gulf Arabic is the smallest class in MADAR. That's not a MADAR quirk — it genuinely reflects the corpus collection process (more Egyptian/Levantine sources were available). This is why I added inverse-frequency class weighting to the training loss.

In [ ]:
counts = df.groupby('dialect')['dialect'].count().reindex(DIALECT_NAMES)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(DIALECT_NAMES, counts.values, color=COLORS, width=0.6, alpha=0.88)

for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            f'{count:,}', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Number of utterances')
ax.set_title('Class Distribution' + (' (Synthetic)' if DATA_IS_SYNTHETIC else ' (MADAR)'))
ax.axhline(counts.mean(), ls='--', color='gray', alpha=0.6, label=f'Mean = {counts.mean():.0f}')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../results/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClass counts:')
print(counts.to_string())
print(f'\nImbalance ratio (max/min): {counts.max()/counts.min():.2f}x')

## 3. Text length distribution

Length matters a lot for this task. Very short utterances (1–3 words) don't give the model enough phonological context to distinguish dialects. The per-length F1 analysis in Notebook 04 shows exactly how much this hurts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Word count distribution per dialect
ax = axes[0]
for label, dialect in enumerate(DIALECT_NAMES):
    subset = df[df['dialect'] == dialect]['n_words']
    ax.hist(subset, bins=20, alpha=0.55, color=COLORS[label], label=dialect)
ax.set_xlabel('Words per utterance')
ax.set_ylabel('Count')
ax.set_title('Word count distribution by dialect')
ax.legend(fontsize=8)

# Box plot
ax = axes[1]
data_by_dialect = [df[df['dialect'] == d]['n_words'].values for d in DIALECT_NAMES]
bp = ax.boxplot(data_by_dialect, patch_artist=True, labels=DIALECT_NAMES)
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Words per utterance')
ax.set_title('Word count box plots')

plt.tight_layout()
plt.show()

print('\nLength statistics:')
print(df.groupby('dialect')['n_words'].describe()[['mean', 'std', 'min', '50%', 'max']].round(1))

## 4. Dialect vocabulary samples

One of the most useful sanity checks: look at actual examples from each class. You can immediately see the phonologically distinctive vocabulary — Gulf *شلونك* vs Egyptian *ازيك* vs Levantine *كيفك* for "how are you". These greetings alone would be enough for a human classifier.

In [ ]:
print('=' * 65)
print('  DIALECT EXAMPLES (3 per class)')
print('=' * 65)
for dialect in DIALECT_NAMES:
    subset = df[df['dialect'] == dialect].sample(min(3, len(df[df['dialect']==dialect])), random_state=42)
    print(f'\n[{dialect}]')
    for _, row in subset.iterrows():
        print(f'  • {row["text"][:80]}')

## 5. Preprocessing pipeline demo

The `clean_arabic_text` function strips diacritics, normalizes alef variants (أ→ا, إ→ا), and optionally normalizes ta marbuta (ة→ه). I kept ta marbuta normalization off by default — it erases a phonological distinction that helps differentiate some dialects.

In [ ]:
from data.madar_preprocessing import clean_arabic_text

test_cases = [
    ('Arabic with diacritics', 'مَرْحَبًا كَيْفَ حَالُكَ'),
    ('Alef variants', 'أحمد إبراهيم آدم الأول'),
    ('Ta marbuta', 'مدينة كبيرة جميلة'),
    ('Mixed noise', 'شَلُونَكَ؟!! هذا تجربة...'),
]

print(f'{"Input":<40} {"Cleaned":<40}')
print('-' * 80)
for description, text in test_cases:
    cleaned = clean_arabic_text(text)
    print(f'{text:<40} {cleaned:<40}  [{description}]')

print(f'\nWith normalize_ta_marbuta=True:')
for _, text in test_cases[2:3]:
    print(f'  Before: {text}')
    print(f'  After:  {clean_arabic_text(text, normalize_ta_marbuta=True)}')

## 6. Summary statistics

A quick summary of what the corpus looks like before we hand it to the graph construction pipeline.

In [ ]:
print('CORPUS SUMMARY')
print('=' * 45)
print(f'  Total examples:  {len(df):,}')
print(f'  Dialect classes: {df["dialect_id"].nunique()}')
print(f'  Avg words/utt:   {df["n_words"].mean():.1f} ± {df["n_words"].std():.1f}')
print(f'  Avg chars/utt:   {df["n_chars"].mean():.1f}')
print(f'  Short utts (<5w):{(df["n_words"] < 5).sum():,} ({(df["n_words"] < 5).mean():.1%})')
print('=' * 45)
print(f'  Data source: {"SYNTHETIC" if DATA_IS_SYNTHETIC else "MADAR"}')
if DATA_IS_SYNTHETIC:
    print('  NOTE: Real MADAR data is required for full replication.')
    print('  See data/README.md for access instructions.')